# Datamine SURVOU Process Example

This notebook demonstrates how to configure and run the **`survou`** process wrapper in `dmstudio`.

## Process Description

## SURVOU

**SURVOU** is a data format conversion process. It requires as input a point file containing the coordinates of each unique point in the survey database and a string segment file containing the links between those points to form strings. These files are output by [SURTAC](<surtac.md>) and [SURVIG](<survig.md>).

**SURVOU** creates an output perimeter file of the type used in mine planning, and surface modeling processes. Points that are stored in the input point file but are not linked within a string will be output to a points file. This allows a file of unconnected points to be supplied as input to the surface digital terrain modelling process [SURTRI](<surtri.md>).

### Input Files:

* **pointin** (Undefined):
  Input point file. This will contain fields PID, X, Y, Z, PSYMBOL, PSYMSZE, P and PERIOD
  (numeric, explicit).
  Required=Yes

* **segin** (Undefined):
  Input string segment file. This will contain PID1, PID2, PVALUE, PTYPE, PCODE, P.
  Required=Yes

### Output Files:

* **perimou** (String):
  Output perimeter file. This will contain the fields XP, YP, ZP, PTN, PVALUE, PTYPE,
  PCODE, P, PSYMBOL, and PSYMSZE. Additional fields found in the input files will be added
  to the output perimeter file.
  Required=Yes

* **pointou** (Undefined):
  Output point file. This will contain fields PID, XP, YP, ZP, PSYMBOL, PSYMSZE, P and
  PERIOD (numeric, explicit).
  Required=Yes

### Fields:

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('survou')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute survou
print("Running survou...")
dm_cmd.survou(
    pointin_i='t_SurfacePointsPt',  # required
    segin_i='t_input_file',  # required
    perimou_o='t_survou_out',  # required
    pointou_o='t_survou_out',  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("survou execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_survou_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")